In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
import numpy as np
import os, time
from tqdm import tqdm

start = time.time()

FOLDER = '/content/drive/MyDrive/Colab Notebooks/德国/Data/Avant-Token'
OUT    = '/content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/fr_all_2015_2025.csv'
REPORT = '/content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/merge_report.csv'

os.makedirs(os.path.dirname(OUT), exist_ok=True)
os.makedirs(os.path.dirname(REPORT), exist_ok=True)

REQUIRED_COLS = ['author','title','date','content','url']

def read_csv_robust(path):
    encs = ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']
    seps = [None, ',', ';', '\t', '|']
    last_err = None
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, engine='python',
                                 encoding=enc, on_bad_lines='skip')
                return df, (sep if sep else 'auto'), enc, None
            except Exception as e:
                last_err = str(e)
    return pd.DataFrame(), 'fail', 'fail', last_err

def normalize_url(u):
    if pd.isna(u):
        return u
    return str(u).strip().lower().rstrip('/')

# 收集 CSV
csv_files = [os.path.join(r, f) for r, _, fs in os.walk(FOLDER)
             for f in fs if f.lower().endswith('.csv')]
print(f"Found {len(csv_files)} CSV files")

records, dfs = [], []

for fp in tqdm(csv_files):
    df, sep, enc, err = read_csv_robust(fp)

    # 清洗列名
    df.columns = [c.strip().replace('\ufeff', '') for c in df.columns]
    df.rename(columns={c: c.lower() for c in df.columns}, inplace=True)

    row_count = len(df)
    has_cols = {c: c in df.columns for c in REQUIRED_COLS}

    records.append({
        'file': fp,
        'rows': row_count,
        'sep': sep,
        'encoding': enc,
        **{f'has_{c}': has_cols[c] for c in REQUIRED_COLS},
        'error': '' if err is None else err[:120]
    })

    if row_count > 0:
        for c in REQUIRED_COLS:
            if c not in df.columns:
                df[c] = None
        dfs.append(df[REQUIRED_COLS])

# 保存读取报告
report_df = pd.DataFrame(records)
report_df.to_csv(REPORT, index=False, encoding='utf-8')
print("Report saved:", REPORT)

if not dfs:
    raise RuntimeError("No valid CSV data loaded.")

# 合并
all_data = pd.concat(dfs, ignore_index=True)

# 清洗空格（注意：不要把 NaN 强行变成字符串 "nan"）
for c in REQUIRED_COLS:
    all_data[c] = all_data[c].astype("string").str.strip()
    all_data.loc[all_data[c].isin(["", "nan", "None", "<NA>"]), c] = pd.NA

# =========================
# 日期解析（真正稳的版本）
# =========================
# 1) 先保证 date 是 string
date_raw = all_data['date'].astype("string").str.strip()

# 2) 统一用矢量化解析，强制 utc=True（这样 tz-aware / tz-naive 都能统一）
#    - tz-aware：保留其时区信息并转到 UTC
#    - tz-naive：当作本地无时区时间，也会被 “加上 UTC”
date_utc = pd.to_datetime(date_raw, errors='coerce', utc=True)

# 3) 转成“无时区”的 datetime64[ns]（关键！）
all_data['date_parsed'] = date_utc.dt.tz_convert(None)

# 过滤区间
start_date = pd.Timestamp('2015-01-01')
end_date   = pd.Timestamp('2025-12-31')

mask_ok = all_data['date_parsed'].between(start_date, end_date, inclusive='both')
mask_bad = all_data['date_parsed'].isna()

final = all_data[mask_ok | mask_bad].copy()

# 去重（按 URL）
final['url_norm'] = final['url'].apply(normalize_url)
final = final.drop_duplicates(subset=['url_norm'])

# 导出
final.to_csv(OUT, index=False, encoding='utf-8')

print(f"Final rows: {len(final):,}")
print(f"Saved to: {OUT}")
print(f"Time: {time.time() - start:.2f}s")
print("date_parsed dtype:", final['date_parsed'].dtype)


Found 741 CSV files


100%|██████████| 741/741 [00:08<00:00, 84.26it/s]


Report saved: /content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/merge_report.csv
Final rows: 695
Saved to: /content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/fr_all_2015_2025.csv
Time: 9.36s
date_parsed dtype: datetime64[ns]


#Tokenisation

In [8]:
!pip -q install spacy
!python -m spacy download fr_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 69.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
import pandas as pd
import spacy
from tqdm import tqdm
import os

# 路径
IN_CSV  = '/content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/fr_all_2015_2025.csv'
OUT_CSV = '/content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/fr_all_2015_2025_tok_spacy.csv'

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# 读取数据
df = pd.read_csv(IN_CSV, encoding='utf-8')

if 'content' not in df.columns:
    raise ValueError("CSV 中没有 content 列")

# 加载法语模型（只启用 tokenizer，提速）
nlp = spacy.load("fr_core_news_sm", disable=["parser","ner","tagger","lemmatizer"])

# 分词函数
def fr_tokenize(text):
    if pd.isna(text):
        return ""
    s = str(text).strip()
    if not s:
        return ""
    doc = nlp(s)
    return " ".join([tok.text for tok in doc])

# 批处理
tqdm.pandas(desc="spaCy 法语分词")
df['content_tok'] = df['content'].progress_apply(fr_tokenize)

# 导出
df.to_csv(OUT_CSV, index=False, encoding='utf-8')

print("已保存：", OUT_CSV)
print(df[['author','title','content_tok']].head(3))


spaCy 法语分词: 100%|██████████| 695/695 [00:29<00:00, 23.26it/s]


已保存： /content/drive/MyDrive/Colab Notebooks/德国/Data/fusion_avantToken/fr_all_2015_2025_tok_spacy.csv
         author                                              title  \
0  Eric Neuhoff        2024, l’année formidable du cinéma français   
1           NaN  L'été 2024 est le plus chaud jamais enregistré...   
2           NaN  Climat: 2024 s'annonce comme la première année...   

                                         content_tok  
0  Alors qu ’ Un p’ tit truc en plus et Le Comte ...  
1  2024 a de fortes chances de devenir la premièr...  
2  Octobre a marqué le 15e mois sur une période d...  
